# Crop Disease Detection — Google Colab Training

EfficientNetB2 (224x224) trained on **lab images + real field images together**, with an
`Unknown` class and open-set rejection so untrained crops are not forced into a trained class.

**Runtime → Change runtime type → T4 GPU**

## Google Drive layout — `My Drive/crop-disease-data/`

| File / folder | Contents | Required |
|---|---|---|
| `Dataset.zip` | lab images, `<ClassName>/*.jpg` | **yes** |
| `field_data.zip` | real field photos, **same class names** | recommended |
| `unknown_leaves.zip` | leaves of crops NOT in the model (guava, papaya, coffee…) | recommended |

A plain folder works instead of a `.zip` (e.g. `crop-disease-data/field_data/`).

## Run
1. Run all cells top to bottom.
2. If Colab disconnects mid-training: **just re-run all cells** — it resumes from the last Drive checkpoint.
3. Copy **all three** files from `model_output/` to `backend/Final_Model/` — they are one matched set:
   `best_model_phase2.weights.h5`, `class_names.json`, `ood_stats.npz`.

## 1. Mount Google Drive & GPU Check

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
assert len(tf.config.list_physical_devices('GPU')) > 0, 'No GPU found! Runtime → Change runtime type → T4 GPU'

## 2. Extract Lab + Field + Unknown Datasets from Google Drive

Unpacks every dataset found in `crop-disease-data/` and locates, at any depth, the folder that
**directly contains the class folders** — zips wrap their contents differently depending on how
they were made, so a fixed-depth guess misses them.

In [ ]:
import os, zipfile, shutil, json, math, random, collections
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.optimizers.schedules import CosineDecay
from PIL import Image

DRIVE_DIR = '/content/gdrive/My Drive/crop-disease-data'
IMG_EXT = ('.jpg', '.jpeg', '.png')

# Name in Drive (a .zip OR a plain folder)  ->  where it is unpacked on the Colab disk.
SOURCES = {
    'Dataset':        '/content/crop-dataset',     # lab images         (REQUIRED)
    'field_data':     '/content/field_data',       # real field photos  (recommended)
    'unknown_leaves': '/content/unknown_leaves',   # non-target leaves  (recommended)
}


def _n_class_dirs(root):
    """How many immediate subfolders directly contain images."""
    if not os.path.isdir(root):
        return 0
    n = 0
    for d in os.listdir(root):
        p = os.path.join(root, d)
        if os.path.isdir(p) and not d.startswith(('__', '.')):
            try:
                if any(f.lower().endswith(IMG_EXT) for f in os.listdir(p)):
                    n += 1
            except OSError:
                pass
    return n


def _find_class_root(root, minimum=1):
    """Score every directory at any depth by how many of its immediate subfolders
    hold images, and return the best one. Handles Dataset.zip/Dataset/<classes>
    just as well as Dataset.zip/<classes>."""
    if not os.path.isdir(root):
        return None
    score = collections.Counter()
    for cur, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith(('__', '.'))]
        if any(f.lower().endswith(IMG_EXT) for f in files):
            score[os.path.dirname(cur)] += 1
    if not score:
        return None
    best, n = score.most_common(1)[0]
    return best if n >= minimum else None


def _fetch(name, dest):
    """Unpack <name>.zip, or copy the folder <name>, from Drive into dest. Re-run safe."""
    if os.path.isdir(dest) and os.listdir(dest):
        print(f'{name:15s} already unpacked at {dest}')
        return dest
    zpath = os.path.join(DRIVE_DIR, name + '.zip')
    fpath = os.path.join(DRIVE_DIR, name)
    if os.path.exists(zpath):
        print(f'{name:15s} extracting {zpath} ...')
        with zipfile.ZipFile(zpath) as z:
            z.extractall(dest)
        return dest
    if os.path.isdir(fpath):
        print(f'{name:15s} copying folder {fpath} ...')
        shutil.copytree(fpath, dest)
        return dest
    print(f'{name:15s} NOT FOUND in {DRIVE_DIR} (looked for {name}.zip and {name}/)')
    return None


for _name, _dest in SOURCES.items():
    _fetch(_name, _dest)

# --- lab dataset (required) -------------------------------------------------
DATASET_DIR = _find_class_root(SOURCES['Dataset'], minimum=10)
if DATASET_DIR is None:
    raise RuntimeError(
        f"No folder with >=10 class folders found under {SOURCES['Dataset']}.\n"
        f'Upload Dataset.zip to {DRIVE_DIR}/ — it must contain one folder per class.')
WORK_DIR = DATASET_DIR                      # section 3 writes Unknown___Unknown in here

# --- field + unknown (optional, but this is what the whole run is for) ------
FIELD_SRC   = _find_class_root(SOURCES['field_data'], minimum=1)
UNKNOWN_SRC = SOURCES['unknown_leaves'] if os.path.isdir(SOURCES['unknown_leaves']) else None

_cls = sorted(d for d in os.listdir(DATASET_DIR)
              if os.path.isdir(os.path.join(DATASET_DIR, d)))
print(f'\nDATASET_DIR = {DATASET_DIR}  ({len(_cls)} class folders)')
print(f'FIELD_SRC   = {FIELD_SRC}'
      f"  ({_n_class_dirs(FIELD_SRC) if FIELD_SRC else 0} class folders)")
print(f'UNKNOWN_SRC = {UNKNOWN_SRC}')
if FIELD_SRC is None:
    print('\n*** No field images. The model will be lab-only and will lose accuracy on real '
          'phone photos. Upload field_data.zip to Drive and re-run this cell. ***')

## 3. Build the Unknown Class from REAL Non-Target Leaves

The `Unknown` class has one job: absorb leaves from crops the model was never trained on, so a
guava leaf is not forced out as "Potato Healthy". **Real** non-target leaves are the part that
does that work — noise and random photos only teach "this is not a plant at all".

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor

TARGET_REAL_LEAVES   = 700      # real leaves of untrained crops  <- the important part
TARGET_RANDOM_PHOTOS = 200      # random real-world photos        <- "not a plant at all"

OUT_DIR = os.path.join(WORK_DIR, 'Unknown___Unknown')
os.makedirs(OUT_DIR, exist_ok=True)


def _count(prefix):
    return len([f for f in os.listdir(OUT_DIR)
                if f.startswith(prefix) and f.lower().endswith(IMG_EXT)])


# --- 1. REAL leaves of untrained crops -------------------------------------
have_leaves = _count('leaf_')

if have_leaves >= 50:
    print(f'Real non-target leaves already present ({have_leaves})')
elif UNKNOWN_SRC and os.path.isdir(UNKNOWN_SRC):
    copied = 0
    for root, _, files in os.walk(UNKNOWN_SRC):
        for f in sorted(files):
            if copied >= TARGET_REAL_LEAVES:
                break
            if f.lower().endswith(IMG_EXT):
                ext = os.path.splitext(f)[1].lower()
                shutil.copy2(os.path.join(root, f),
                             os.path.join(OUT_DIR, f'leaf_{copied:04d}{ext}'))
                copied += 1
    have_leaves = copied
    print(f'Copied {copied} REAL non-target leaf images from {UNKNOWN_SRC}')
else:
    print('*** WARNING: no non-target leaf dataset found. ***')
    print(f'    Upload unknown_leaves.zip to {DRIVE_DIR}/ containing leaves of crops this')
    print('    model does NOT cover (guava, papaya, coffee, tea, sugarcane, betel...).')
    print('    Without them the Unknown class cannot reject an untrained crop -- a guava')
    print('    leaf will keep coming out as "Potato Healthy" with high confidence.')

# --- 2. random real-world photos (threaded) --------------------------------
have_random = _count('picsum_')
if have_random < TARGET_RANDOM_PHOTOS // 2:
    sess = requests.Session()

    def _grab(i):
        try:
            r = sess.get(f'https://picsum.photos/seed/unknown{i}/300/300',
                         timeout=5, allow_redirects=True)
            if r.status_code == 200 and len(r.content) > 1000:
                with open(os.path.join(OUT_DIR, f'picsum_{i:04d}.jpg'), 'wb') as f:
                    f.write(r.content)
                return True
        except Exception:
            pass
        return False

    with ThreadPoolExecutor(max_workers=24) as ex:
        ok = sum(ex.map(_grab, range(TARGET_RANDOM_PHOTOS * 2)))
    print(f'Downloaded {ok} random real-world photos')
else:
    print(f'Random photos already present ({have_random})')

print(f'\nUnknown so far -- real leaves: {_count("leaf_")}, random photos: {_count("picsum_")}')

## 4. Top Up Unknown with Synthetic Images (Noise + Leaf-Like Textures)

Filler only — capped at 35% of the class so synthetic images can never dominate it.

In [ ]:
from PIL import ImageDraw

OUT_DIR = os.path.join(WORK_DIR, 'Unknown___Unknown')
rng = random.Random(42)

# Total size of the Unknown class. Synthetic images only fill the gap left after
# the real non-target leaves and random photos from section 3 -- they are cheap
# insurance, not the mechanism that rejects an untrained crop.
TARGET_UNKNOWN_TOTAL = 1000
SYNTH_MAX_SHARE = 0.35          # never let synthetic images dominate the class

def _total():
    return len([f for f in os.listdir(OUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

real_now = _total()
gap = max(0, TARGET_UNKNOWN_TOTAL - real_now)
gap = min(gap, int(TARGET_UNKNOWN_TOTAL * SYNTH_MAX_SHARE))
n_noise = gap // 3
n_texture = gap - n_noise
print(f'Unknown has {real_now} real images; adding {n_noise} noise + {n_texture} texture')

if n_noise > 0 and len([f for f in os.listdir(OUT_DIR) if f.startswith('gn_')]) < n_noise:
    sigmas = [8, 20, 40, 80, 160]
    per = max(1, n_noise // len(sigmas))
    for sigma in sigmas:
        for i in range(per):
            arr = np.clip(np.random.randn(224, 224, 3) * sigma + 128, 0, 255).astype(np.uint8)
            Image.fromarray(arr).save(os.path.join(OUT_DIR, f'gn_{sigma}_{i:04d}.jpg'))
    print(f'Generated {per * len(sigmas)} Gaussian noise images')

if n_texture > 0 and len([f for f in os.listdir(OUT_DIR) if f.startswith('lt_')]) < n_texture:
    for i in range(n_texture):
        bg = rng.randint(30, 90)
        img = Image.new('RGB', (224, 224), (bg, bg + 20, bg))
        draw = ImageDraw.Draw(img)
        for _ in range(rng.randint(3, 12)):
            cx, cy = rng.randint(30, 194), rng.randint(30, 194)
            rx, ry = rng.randint(15, 100), rng.randint(10, 60)
            g = rng.randint(60, 200)
            r = rng.randint(20, min(g, 110))
            b = rng.randint(15, 55)
            draw.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=(r, g, b))
        for _ in range(rng.randint(0, 4)):
            x1, y1, x2, y2 = [rng.randint(0, 224) for _ in range(4)]
            draw.line([(x1, y1), (x2, y2)],
                      fill=(rng.randint(40, 80), rng.randint(60, 120), rng.randint(20, 50)),
                      width=rng.randint(1, 3))
        img.save(os.path.join(OUT_DIR, f'lt_{i:04d}.jpg'))
    print(f'Generated {n_texture} leaf-like texture images')

_leaves = len([f for f in os.listdir(OUT_DIR) if f.startswith('leaf_')])
print(f'\nTotal Unknown___Unknown images: {_total()}  (real non-target leaves: {_leaves})')
if _leaves < 100:
    print('WARNING: fewer than 100 real non-target leaves. Untrained crops such as guava')
    print('         are likely to still be predicted as a trained class.')

## 5. Training Configuration

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
EPOCHS_PHASE1 = 10
EPOCHS_PHASE2 = 15
LR_PHASE1 = 5e-4
LR_PHASE2 = 1e-4
DROPOUT_RATE = 0.5
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 1e-4

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print('Config set.')

# ---- Speed knobs ----------------------------------------------------------
# BATCH_SIZE 64 halves the number of steps and keeps the GPU fed. Combined with
# mixed precision it is the single biggest wall-clock win after the data loader.
MIXED_PRECISION = True    # float16 compute on tensor cores (~1.5-2x on T4/P100)
LOADER_WORKERS  = 8       # parallel image decode+augment. Default is 1 (!).
MAX_QUEUE       = 32      # batches pre-fetched ahead of the GPU

if MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision:', tf.keras.mixed_precision.global_policy().name)


## 6. Stratified Split (70/15/15) + Merge Real Field Images

Lab and field images are split separately, then **merged into the same train/val/test folders**,
so every class is learned from both domains at once. The field test slice is *also* mirrored into
`data/field_test` so section 15 can score real-world performance on its own.

In [ ]:
base_dir = '/content/data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')


def _class_dirs(root):
    """Folders that directly contain images - real class folders, not wrappers."""
    out = []
    for d in sorted(os.listdir(root)):
        p = os.path.join(root, d)
        if not os.path.isdir(p) or d.startswith(('.', '__')):
            continue
        if any(f.lower().endswith(IMG_EXT) for f in os.listdir(p)):
            out.append(d)
    return out


classes = _class_dirs(DATASET_DIR)
if len(classes) < 10:
    raise RuntimeError(
        f'Only {len(classes)} class folders found in "{DATASET_DIR}": {classes}\n'
        f'Section 2 resolved the wrong level - DATASET_DIR must be the folder that '
        f'DIRECTLY contains the class folders (Apple__Black_Rot, ...).')
print(f'{len(classes)} class folders detected.\n')

# --- lab images ------------------------------------------------------------
for cls in classes:
    src = os.path.join(DATASET_DIR, cls)
    images = [f for f in os.listdir(src) if f.lower().endswith(IMG_EXT)]
    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * 0.70)
    n_val = int(n_total * 0.15)
    train_imgs = images[:n_train]
    val_imgs = images[n_train:n_train + n_val]
    test_imgs = images[n_train + n_val:]

    for split_dir, split_imgs in [
        (os.path.join(train_dir, cls), train_imgs),
        (os.path.join(val_dir, cls), val_imgs),
        (os.path.join(test_dir, cls), test_imgs),
    ]:
        os.makedirs(split_dir, exist_ok=True)
        for img in split_imgs:
            shutil.copy2(os.path.join(src, img), os.path.join(split_dir, img))

    print(f'{cls:45s} {len(train_imgs):4d} train, {len(val_imgs):4d} val, {len(test_imgs):4d} test')
print(f'\nTotal classes: {len(classes)}')

# ---------------------------------------------------------------------------
# Mix in REAL FIELD images to close the lab->field domain gap. Field photos live
# in <FIELD_DIR>/<ClassName>/*.jpg using the SAME class names as the lab set.
# Split 70/15/15 like the lab data; the field TEST slice is mirrored into
# data/field_test for a separate real-world score (see section 15).
#
# Classes with fewer than FIELD_MIN_SPLIT field images route ALL of them into
# training: those few real field scenes are too precious to spend on eval, and a
# field-only score on 1-2 images would be pure noise. They are still tested on
# their lab images in the normal test set.
# ---------------------------------------------------------------------------
FIELD_DIR = FIELD_SRC
FIELD_MIN_SPLIT = 20
field_test_dir = os.path.join(base_dir, 'field_test')
print(f'\nField images dir: {FIELD_DIR}  (exists: {bool(FIELD_DIR) and os.path.isdir(FIELD_DIR)})')

if FIELD_DIR and os.path.isdir(FIELD_DIR):
    print('\nMerging real field images:')
    total_field = 0
    for cls in classes:
        fsrc = os.path.join(FIELD_DIR, cls)
        if not os.path.isdir(fsrc):
            continue
        fimgs = [f for f in os.listdir(fsrc) if f.lower().endswith(IMG_EXT)]
        if not fimgs:
            continue
        random.shuffle(fimgs)
        nf = len(fimgs)
        if nf < FIELD_MIN_SPLIT:
            parts = [(os.path.join(train_dir, cls), fimgs)]
            split_note = f'{nf} tr (all -> train, too few to split)'
        else:
            n_tr = int(nf * 0.70); n_va = int(nf * 0.15)
            parts = [
                (os.path.join(train_dir, cls), fimgs[:n_tr]),
                (os.path.join(val_dir, cls),   fimgs[n_tr:n_tr + n_va]),
                (os.path.join(test_dir, cls),  fimgs[n_tr + n_va:]),
                (os.path.join(field_test_dir, cls), fimgs[n_tr + n_va:]),  # field-only mirror
            ]
            split_note = f'{n_tr} tr / {n_va} va / {nf - n_tr - n_va} te'
        for split_dir, split_imgs in parts:
            os.makedirs(split_dir, exist_ok=True)
            for img in split_imgs:
                shutil.copy2(os.path.join(fsrc, img), os.path.join(split_dir, 'field_' + img))
        total_field += nf
        print(f'  {cls:45s} +{nf:4d} field ({split_note})')
    print(f'Merged {total_field} field images.')
    if total_field == 0:
        print('WARNING: field dir found but 0 images merged - class names likely differ '
              'between the lab and field datasets.')
else:
    print('\n[no field images] Upload field_data.zip to Drive with <ClassName>/*.jpg folders '
          'matching the lab class names. Training on lab images only for now.')

## 7. Balance Classes — Oversample Minorities, Undersample Majorities

Undersampling deletes files, so real field photos (`field_*`) and real non-target leaves
(`leaf_*`) are **protected** — synthetic filler is dropped first.

In [ ]:
from PIL import Image, ImageEnhance, ImageFilter
import numpy as np

train_counts = {}
for cls in classes:
    cls_dir = os.path.join(train_dir, cls)
    if os.path.isdir(cls_dir):
        n = len([f for f in os.listdir(cls_dir) if f.lower().endswith(IMG_EXT)])
        train_counts[cls] = n

counts = sorted(train_counts.values())
median_count = counts[len(counts) // 2]
floor_count = int(median_count * 1.2)   # oversample minorities UP to here
ceil_count  = int(median_count * 2.0)   # undersample majorities DOWN to here (caps ratio ~2:1)

print(f'Image counts -- min: {counts[0]}, median: {median_count}, max: {counts[-1]}')
print(f'Balancing band: floor={floor_count}, ceil={ceil_count}')

rng = np.random.RandomState(42)

# Undersampling deletes files from the working copy. Some images are far more
# valuable than others and must never be the ones dropped:
#   field_*  real field photos       - the scarcest data in the run
#   leaf_*   real non-target leaves  - the only thing that teaches Unknown to
#                                      reject an untrained crop such as guava
# Synthetic filler is dropped first, then ordinary lab images.
PROTECTED = ('field_', 'leaf_')
DROP_FIRST = ('gn_', 'lt_', 'picsum_', 'aug_')

for cls in classes:
    cls_dir = os.path.join(train_dir, cls)
    if not os.path.isdir(cls_dir):
        continue
    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(IMG_EXT)]
    n = len(imgs)
    if n == 0:
        continue

    # --- Undersample majority classes down to the ceiling ---
    if n > ceil_count:
        need_drop = n - ceil_count
        synthetic = [f for f in imgs if f.startswith(DROP_FIRST)]
        ordinary  = [f for f in imgs if not f.startswith(DROP_FIRST + PROTECTED)]
        pool = list(rng.permutation(synthetic)) + list(rng.permutation(ordinary))
        drop = pool[:need_drop]
        for f in drop:
            os.remove(os.path.join(cls_dir, f))
        kept_protected = sum(f.startswith(PROTECTED) for f in imgs)
        note = f' [{kept_protected} protected kept]' if kept_protected else ''
        print(f'  {cls:45s} {n:4d} -> {n - len(drop):4d}  (-{len(drop)} undersampled){note}')
        continue

    # --- Oversample minority classes up to the floor ---
    # Never more than ~5x the real images: repeating the same few photos further
    # just teaches the model to memorise them.
    if n < floor_count:
        need = min(floor_count - n, n * 5)
        for i in range(need):
            src_img = Image.open(os.path.join(cls_dir, imgs[i % len(imgs)])).convert('RGB')
            src_img = src_img.resize(IMG_SIZE, Image.LANCZOS)
            if rng.random() < 0.5:
                src_img = src_img.transpose(Image.FLIP_LEFT_RIGHT)
            if rng.random() < 0.4:
                src_img = ImageEnhance.Brightness(src_img).enhance(rng.uniform(0.7, 1.3))
            if rng.random() < 0.4:
                src_img = ImageEnhance.Contrast(src_img).enhance(rng.uniform(0.7, 1.3))
            if rng.random() < 0.3:
                src_img = src_img.rotate(rng.uniform(-25, 25), resample=Image.BICUBIC,
                                         fillcolor=(0, 0, 0))
            ext = os.path.splitext(imgs[i % len(imgs)])[1]
            src_img.save(os.path.join(cls_dir, f'aug_{i:04d}{ext}'))
        print(f'  {cls:45s} {n:4d} -> {n + need:4d}  (+{need} augmented)')

print('\nBalancing complete.')


# --- Pre-resize every split to IMG_SIZE once -------------------------------
# Decoding a 4000x3000 phone photo and resizing it to 224x224 costs more CPU
# than the forward pass costs GPU -- and it is paid again every single epoch.
# Doing it once on disk is the difference between a GPU that waits and one that
# works. Lab images are already small, so this mostly pays off on field photos.
PRE_RESIZE = True
if PRE_RESIZE:
    _resized = _already = 0
    for _split in (train_dir, val_dir, test_dir, field_test_dir):
        if not os.path.isdir(_split):
            continue
        for _cls in os.listdir(_split):
            _cdir = os.path.join(_split, _cls)
            if not os.path.isdir(_cdir):
                continue
            for _f in os.listdir(_cdir):
                if not _f.lower().endswith(IMG_EXT):
                    continue
                _p = os.path.join(_cdir, _f)
                try:
                    with Image.open(_p) as _im:
                        if _im.size == IMG_SIZE:
                            _already += 1
                            continue
                        _out = _im.convert('RGB').resize(IMG_SIZE, Image.LANCZOS)
                    if _p.lower().endswith('.png'):
                        _out.save(_p)
                    else:
                        _out.save(_p, quality=95)
                    _resized += 1
                except Exception:
                    pass
    print(f'\nPre-resized {_resized} images to {IMG_SIZE} ({_already} already correct)')


## 8. Compute Class Weights

In [ ]:
import math

class_counts = {}
for i, cls in enumerate(classes):
    cls_dir = os.path.join(train_dir, cls)
    n = len([f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if n > 0:
        class_counts[i] = n

if not class_counts:
    raise RuntimeError('No training images found in any class! Check dataset path.')

# Gentle sqrt-damped weights. Counts are already balanced into a band above, so
# these stay near 1.0 (~1.0-1.5). The old max_count/n scheme (1-6) on top of the
# resampling double-corrected and destabilised training.
mean_count = sum(class_counts.values()) / len(class_counts)
class_weight = {i: math.sqrt(mean_count / n) for i, n in class_counts.items()}

print('Class weights (sqrt-damped):')
for i, cls in enumerate(classes):
    n = class_counts.get(i, 0)
    if n == 0:
        print(f'  {cls:45s} {"SKIPPED":>4s} images')
    else:
        print(f'  {cls:45s} {n:4d} images  weight={class_weight[i]:5.2f}')


## 9. Data Augmentation + Field-Realism Augmentation

`field_augment` degrades clean lab images the way a phone camera in a field does — blur, sensor
noise, directional shadow — so lab images teach the model what field images look like.

In [ ]:
import cv2

def field_augment(img):
    """Make a clean lab image look more like a field photo: random blur, sensor
    noise, and a soft directional shadow. Runs AFTER the geometric transforms
    and BEFORE EfficientNet preprocessing. img is a HxWx3 array in [0, 255]."""
    x = img.astype(np.float32)
    r = np.random.random
    if r() < 0.35:                                   # out-of-focus / motion blur
        ksize = int(np.random.choice([3, 5]))
        x = cv2.GaussianBlur(x, (ksize, ksize), 0)
    if r() < 0.30:                                   # sensor noise
        x = x + np.random.normal(0, np.random.uniform(4, 14), x.shape)
    if r() < 0.30:                                   # soft directional shadow
        h, w = x.shape[:2]
        strength = np.random.uniform(0.5, 0.85)
        if r() < 0.5:
            ramp = np.linspace(strength, 1.0, w)
            shadow = np.tile(ramp, (h, 1))
        else:
            ramp = np.linspace(strength, 1.0, h)
            shadow = np.tile(ramp[:, None], (1, w))
        x = x * shadow[:, :, None]
    x = np.clip(x, 0, 255)
    return tf.keras.applications.efficientnet.preprocess_input(x)

def get_train_datagen():
    return tf.keras.preprocessing.image.ImageDataGenerator(
        rotation_range=40,
        width_shift_range=0.25,
        height_shift_range=0.25,
        shear_range=0.2,
        zoom_range=0.3,
        brightness_range=(0.6, 1.4),
        channel_shift_range=30.0,
        horizontal_flip=True,
        vertical_flip=False,
        fill_mode='reflect',
        preprocessing_function=field_augment,
    )

def get_eval_datagen():
    return tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    )


## 10. Create Data Generators

In [ ]:
train_gen = get_train_datagen().flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)
val_gen = get_eval_datagen().flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_gen = get_eval_datagen().flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'Training samples: {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Test samples: {test_gen.samples}')

# ---------------------------------------------------------------------------
# Keras loads and augments images on ONE thread by default (workers=1), so the
# GPU sits idle waiting for batches. cv2, numpy and PIL all release the GIL, so
# threads scale here; multiprocessing is avoided because forked workers inherit
# the same numpy RNG state and would repeat identical augmentations.
# ---------------------------------------------------------------------------
for _g in (train_gen, val_gen, test_gen):
    try:
        _g.workers = LOADER_WORKERS
        _g.use_multiprocessing = False
        _g.max_queue_size = MAX_QUEUE
    except AttributeError:
        pass   # older Keras: workers are passed to model.fit instead
print(f'Loader workers: {getattr(train_gen, "workers", "n/a")}, '
      f'queue: {getattr(train_gen, "max_queue_size", "n/a")}')


## 11. Build Model (EfficientNetB2)

In [ ]:
base_model = EfficientNetB2(
    include_top=False, weights='imagenet',
    input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(512, activation='relu', name='dense_hidden',
                 kernel_regularizer=regularizers.l2(WEIGHT_DECAY))(x)
x = layers.Dropout(DROPOUT_RATE)(x)
outputs = layers.Dense(
    len(classes), activation='softmax', name='dense_output',
    dtype='float32',   # softmax stays fp32 under mixed precision
    kernel_regularizer=regularizers.l2(WEIGHT_DECAY)
)(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

## 12. Phase 1 — Train Top Layers (checkpointed to Drive)

Checkpoints on **macro-F1**, not `val_accuracy`: macro-F1 weights every class equally, so the
saved model cannot win by favouring the crops that happen to have the most images.

In [ ]:
from sklearn.metrics import f1_score

# ---------------------------------------------------------------------------
# Checkpoint on macro-F1, not val_accuracy: macro-F1 weights every class equally,
# so the saved model cannot win by favouring the crops with the most images.
#
# Keras >= 3 computes macro-F1 as a normal metric during the validation pass the
# model already runs. The callback below is the fallback for older Keras, and it
# costs a SECOND full prediction over the validation set every epoch.
# ---------------------------------------------------------------------------
USE_F1_METRIC = hasattr(tf.keras.metrics, 'F1Score')
F1_MONITOR = 'val_macro_f1'


def build_metrics():
    m = ['accuracy']
    if USE_F1_METRIC:
        m.append(tf.keras.metrics.F1Score(average='macro', name='macro_f1'))
    return m


def f1_checkpoint(val_gen, filepath):
    """ModelCheckpoint on macro-F1 when Keras can compute it, else the callback."""
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    if USE_F1_METRIC:
        return tf.keras.callbacks.ModelCheckpoint(
            filepath, monitor=F1_MONITOR, mode='max',
            save_best_only=True, save_weights_only=True, verbose=1)
    return MacroF1Checkpoint(val_gen, filepath)


class MacroF1Checkpoint(tf.keras.callbacks.Callback):
    """Fallback for Keras < 3: an extra prediction pass over validation."""
    def __init__(self, val_gen, filepath):
        super().__init__()
        self.val_gen = val_gen
        self.filepath = filepath
        self.best = -1.0
        os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        self.val_gen.reset()
        y_true = self.val_gen.classes
        y_pred = np.argmax(self.model.predict(self.val_gen, verbose=0), axis=1)
        f1 = f1_score(y_true, y_pred, average='macro')
        if logs is not None:
            logs['val_macro_f1'] = f1
        marker = ''
        if f1 > self.best:
            self.best = f1
            self.model.save_weights(self.filepath)
            marker = '  <-- saved (best macro-F1)'
        print(f'  val_macro_f1: {f1:.4f}{marker}')

# Paths on Google Drive (persist across Colab disconnects)
DRIVE_CKPT = os.path.join(DRIVE_DIR, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

PHASE1_WEIGHTS = os.path.join(DRIVE_CKPT, 'best_model_phase1.weights.h5')
PHASE2_WEIGHTS = os.path.join(DRIVE_CKPT, 'best_model_phase2.weights.h5')
PHASE2_EPOCHS_FILE = os.path.join(DRIVE_CKPT, 'phase2_epochs_done.txt')

history1 = None
history2 = None

if os.path.exists(PHASE2_WEIGHTS):
    print(f'Found Phase 2 weights on Drive: {PHASE2_WEIGHTS}')
    model.load_weights(PHASE2_WEIGHTS)
    print('Phase 1 SKIPPED -- loading Phase 2 weights directly')
elif os.path.exists(PHASE1_WEIGHTS):
    print(f'Found Phase 1 weights on Drive: {PHASE1_WEIGHTS}')
    model.load_weights(PHASE1_WEIGHTS)
    print('Phase 1 SKIPPED -- proceeding to Phase 2')
else:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_PHASE1),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=build_metrics()
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=7, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
        ),
        f1_checkpoint(val_gen, PHASE1_WEIGHTS),
    ]

    print('Phase 1: Training top layers...')
    history1 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS_PHASE1, callbacks=callbacks,
        class_weight=class_weight, verbose=1
    )
    model.load_weights(PHASE1_WEIGHTS)  # carry the best macro-F1 epoch into Phase 2

## 13. Phase 2 — Fine-Tune Entire Model (Resume-Safe)

If Colab disconnects, re-run every cell: this picks up from the Drive checkpoint at the epoch
recorded in `phase2_epochs_done.txt`.

In [ ]:
# ---- Resume logic ----
epochs_done = 0
if os.path.exists(PHASE2_EPOCHS_FILE):
    with open(PHASE2_EPOCHS_FILE) as f:
        try:
            epochs_done = int(f.read().strip())
        except ValueError:
            epochs_done = 0
if os.path.exists(PHASE2_WEIGHTS):
    print('Found existing Phase 2 checkpoint on Drive')
    model.load_weights(PHASE2_WEIGHTS)
    print(f'Resuming: {epochs_done}/{EPOCHS_PHASE2} epochs completed')

remaining = EPOCHS_PHASE2 - epochs_done

if remaining > 0:
    base_model.trainable = True
    for layer in base_model.layers[:100]:
        layer.trainable = False

    total_steps = len(train_gen) * remaining
    cosine_schedule = CosineDecay(
        initial_learning_rate=LR_PHASE2,
        decay_steps=total_steps,
        alpha=0.01
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(cosine_schedule),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=build_metrics()
    )

    class EpochTracker(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            total = epochs_done + epoch + 1
            with open(PHASE2_EPOCHS_FILE, 'w') as f:
                f.write(str(total))
            if total % 5 == 0:
                print(f'Checkpoint synced: {total}/{EPOCHS_PHASE2} epochs')

    callbacks2 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=7, restore_best_weights=True
        ),
        f1_checkpoint(val_gen, PHASE2_WEIGHTS),
        EpochTracker(),
    ]

    print(f'Phase 2: Fine-tuning {remaining} epochs ({epochs_done} already done)...')
    history2 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=remaining, callbacks=callbacks2,
        class_weight=class_weight, verbose=1
    )
    model.load_weights(PHASE2_WEIGHTS)
    print('Phase 2 complete! Best weights saved to Drive.')
else:
    print(f'Phase 2 already completed ({epochs_done}/{EPOCHS_PHASE2} epochs)')

## 14. Evaluate on Test Set — Overall Metrics

In [ ]:
def clean_name(label):
    name = label.replace('___', ' ').replace('__', ' ').replace('_', ' ')
    return ' '.join(name.split())

test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f'Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test loss: {test_loss:.4f}')

all_preds = model.predict(test_gen, verbose=0)
pred_classes = np.argmax(all_preds, axis=1)
true_classes = test_gen.classes

## 15. Per-Class Metrics + Field-Only Evaluation

The **field-only** score is the number that matters for a phone-camera app: the gap between it and
the lab test score is the remaining domain gap.

In [ ]:
from sklearn.metrics import classification_report

target_names = [clean_name(c) for c in classes]
print(classification_report(true_classes, pred_classes, target_names=target_names, digits=3))

# ---- Field-only accuracy: how the model does on REAL field photos ----
_field_test = os.path.join(base_dir, "field_test")
if os.path.isdir(_field_test) and any(os.scandir(_field_test)):
    field_gen = get_eval_datagen().flow_from_directory(
        _field_test, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False, classes=classes
    )
    if field_gen.samples > 0:
        from sklearn.metrics import f1_score as _f1
        fp = np.argmax(model.predict(field_gen, verbose=0), axis=1)
        ft = field_gen.classes
        print(f'\n=== FIELD-ONLY test ({field_gen.samples} real images) ===')
        print(f'Field accuracy : {(fp == ft).mean():.3f}')
        print(f'Field macro-F1 : {_f1(ft, fp, average="macro"):.3f}')
        print('Compare with the lab test numbers above -- the gap is your remaining domain gap.')
else:
    print('\n[field-only eval skipped] No field_test split -- add field images to measure real-world accuracy.')


## 15b. Untrained-Crop Rejection Test

In [ ]:
# Mirrors the unknown-detection rule in backend/main.py so this score matches
# deployed behaviour. Keep the two in sync if you tune the thresholds.
UNKNOWN_CLASS = 'Unknown___Unknown'


def backend_decision(probs):
    """Return (predicted_class, is_unknown) using the backend's exact rule."""
    order = np.argsort(probs)[::-1]
    top1, top2 = float(probs[order[0]]), float(probs[order[1]])
    margin = (top1 - top2) * 100
    p = np.clip(probs, 1e-12, 1.0)
    entropy = -np.sum(p * np.log(p)) / np.log(len(p))
    is_unknown = (
        top1 < 0.30
        or (top1 < 0.50 and margin < 5)
        or (entropy > 0.90)
    )
    cls = classes[order[0]]
    if cls == UNKNOWN_CLASS:
        is_unknown = True
    return cls, is_unknown, top1, entropy


def check_rejection(folder, label=None, limit=200):
    """Run every image in `folder` (recursively) and report how many are rejected."""
    paths = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            if f.lower().endswith(IMG_EXT):
                paths.append(os.path.join(root, f))
    paths = paths[:limit]
    if not paths:
        print(f'No images found in {folder}')
        return
    batch = []
    for p in paths:
        im = Image.open(p).convert('RGB').resize(IMG_SIZE, Image.LANCZOS)
        batch.append(np.array(im, dtype=np.float32))
    x = tf.keras.applications.efficientnet.preprocess_input(np.stack(batch))
    preds = model.predict(x, verbose=0)

    rejected, forced = 0, {}
    for pr in preds:
        cls, is_unk, top1, ent = backend_decision(pr)
        if is_unk:
            rejected += 1
        else:
            forced[cls] = forced.get(cls, 0) + 1

    name = label or os.path.basename(folder.rstrip('/'))
    print(f'\n=== Untrained-crop rejection: {name} ({len(paths)} images) ===')
    print(f'Routed to Unknown : {rejected}/{len(paths)}  ({rejected / len(paths) * 100:.1f}%)')
    if forced:
        print('Confidently misclassified as:')
        for c, n in sorted(forced.items(), key=lambda kv: -kv[1])[:8]:
            print(f'   {clean_name(c):40s} {n:4d}')
    return rejected / len(paths)


# --- 1. Unknown recall on the held-out test set -----------------------------
if UNKNOWN_CLASS in classes:
    ui = classes.index(UNKNOWN_CLASS)
    mask = true_classes == ui
    if mask.sum():
        recall = (pred_classes[mask] == ui).mean()
        # How often a KNOWN class is wrongly sent to Unknown (the cost of rejecting)
        known = ~mask
        false_unknown = (pred_classes[known] == ui).mean()
        print(f'Unknown recall on test set : {recall:.3f}  ({mask.sum()} images)')
        print(f'Known leaves sent to Unknown: {false_unknown:.3f}  <- keep this low')
else:
    print(f'No {UNKNOWN_CLASS} class in this model - untrained crops cannot be rejected.')

# --- 2. Rejection rate on a folder of genuinely untrained crops --------------
# Put a folder of leaves from crops NOT in this model (guava,
# papaya, coffee...) held out from training, and point this at it.
HOLDOUT_UNTRAINED = None    # e.g. '/content/gdrive/My Drive/crop-disease-data/guava_holdout'

if HOLDOUT_UNTRAINED and os.path.isdir(HOLDOUT_UNTRAINED):
    check_rejection(HOLDOUT_UNTRAINED, 'held-out untrained crops')
else:
    print('\n[skipped] Set HOLDOUT_UNTRAINED to a folder of untrained-crop leaves')
    print('          (kept OUT of training) to measure real rejection performance.')

## 15c. Open-Set Rejection — Reject ANY Untrained Crop

Softmax cannot solve this on its own: an untrained crop *must* be assigned to one of the trained
classes. But an untrained species lands far from every trained class centroid in feature space,
even when softmax looks confident — and that distance generalises to species never collected.

In [ ]:
# ---------------------------------------------------------------------------
# Open-set rejection in FEATURE space.
#
# Softmax cannot solve this: an untrained crop must be assigned to one of the
# trained classes, and the model is often genuinely confident about it. But an
# untrained species lands FAR from every trained class centroid in the
# penultimate feature space, even when its softmax output looks certain. That
# distance generalises to species that were never in the Unknown class.
# ---------------------------------------------------------------------------
OUTPUT_DIR = '/content/model_output'
TARGET_KNOWN_ACCEPT = 0.95   # keep this fraction of genuine known-crop leaves

embed_model = tf.keras.Model(model.input, model.get_layer('dense_hidden').output)


def _embed(gen_or_array):
    e = embed_model.predict(gen_or_array, verbose=0)
    e = e.astype(np.float32)   # dense_hidden is fp16 under mixed precision
    return e / (np.linalg.norm(e, axis=1, keepdims=True) + 1e-9)


# --- 1. one centroid per TRAINED class (Unknown excluded) -------------------
cent_gen = get_eval_datagen().flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False, classes=classes)
print('Embedding the training set for centroids...')
train_emb = _embed(cent_gen)
train_lab = np.array(cent_gen.classes)

known_idx = [i for i, c in enumerate(classes) if c != UNKNOWN_CLASS]
centroids, centroid_classes = [], []
for i in known_idx:
    m = train_emb[train_lab == i]
    if len(m) == 0:
        continue
    c = m.mean(axis=0)
    centroids.append(c / (np.linalg.norm(c) + 1e-9))
    centroid_classes.append(classes[i])
centroids = np.stack(centroids).astype(np.float32)
print(f'{len(centroids)} class centroids, dim {centroids.shape[1]}')


def ood_score(emb_norm):
    """Highest cosine similarity to any trained-class centroid. Low = unfamiliar."""
    return (emb_norm @ centroids.T).max(axis=1)


# --- 2. calibrate the threshold on VALIDATION known classes ----------------
val_eval = get_eval_datagen().flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False, classes=classes)
val_emb = _embed(val_eval)
val_known = np.isin(np.array(val_eval.classes), known_idx)
val_scores = ood_score(val_emb)[val_known]
OOD_THRESHOLD = float(np.percentile(val_scores, (1 - TARGET_KNOWN_ACCEPT) * 100))
print(f'\nOOD threshold = {OOD_THRESHOLD:.4f}  '
      f'(accepts {TARGET_KNOWN_ACCEPT:.0%} of known validation leaves)')


# --- 3. combined decision: softmax rule OR feature-distance rule ------------
def openset_decision(probs, emb_norm):
    """Backend softmax rule OR feature-distance rule. emb_norm is one L2-normalised
    embedding. Returns (class, is_unknown, confidence, ood_score)."""
    cls, is_unk, top1, _ent = backend_decision(probs)
    score = float(ood_score(np.atleast_2d(emb_norm))[0])
    if score < OOD_THRESHOLD:
        is_unk = True
    return cls, is_unk, top1, score


# --- 4. what it costs and what it buys -------------------------------------
test_emb = _embed(test_gen)
test_scores = ood_score(test_emb)
test_known = np.isin(np.array(test_gen.classes), known_idx)

print(f'\n=== Open-set rejection on the test set ===')
print(f'Known leaves wrongly rejected : {(test_scores[test_known] < OOD_THRESHOLD).mean():.3f}'
      f'   <- the cost, keep low')
if (~test_known).sum():
    print(f'Unknown-class images rejected : {(test_scores[~test_known] < OOD_THRESHOLD).mean():.3f}')


def check_rejection_openset(folder, label=None, limit=200):
    """Rejection rate on a folder of untrained-crop photos, using BOTH rules."""
    paths = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            if f.lower().endswith(IMG_EXT):
                paths.append(os.path.join(root, f))
    paths = paths[:limit]
    if not paths:
        print(f'No images found in {folder}')
        return
    batch = [np.array(Image.open(p).convert('RGB').resize(IMG_SIZE, Image.LANCZOS),
                      dtype=np.float32) for p in paths]
    x = tf.keras.applications.efficientnet.preprocess_input(np.stack(batch))
    probs = model.predict(x, verbose=0)
    embs = _embed(x)
    scores = ood_score(embs)

    by_softmax = by_distance = rejected = 0
    forced = {}
    for pr, sc in zip(probs, scores):
        cls, unk_soft, _, _ = backend_decision(pr)
        unk_dist = sc < OOD_THRESHOLD
        by_softmax += unk_soft
        by_distance += unk_dist
        if unk_soft or unk_dist:
            rejected += 1
        else:
            forced[cls] = forced.get(cls, 0) + 1

    name = label or os.path.basename(folder.rstrip('/'))
    n = len(paths)
    print(f'\n=== Open-set rejection: {name} ({n} images) ===')
    print(f'Rejected overall      : {rejected}/{n}  ({rejected / n * 100:.1f}%)')
    print(f'  caught by softmax   : {by_softmax}/{n}')
    print(f'  caught by distance  : {by_distance}/{n}   <- the part that generalises')
    if forced:
        print('Still misclassified as:')
        for c, k in sorted(forced.items(), key=lambda kv: -kv[1])[:8]:
            print(f'   {clean_name(c):40s} {k:4d}')
    return rejected / n


if HOLDOUT_UNTRAINED and os.path.isdir(HOLDOUT_UNTRAINED):
    check_rejection_openset(HOLDOUT_UNTRAINED, 'held-out untrained crops')
else:
    print('\n[holdout skipped] Set HOLDOUT_UNTRAINED in the previous section to measure '
          'rejection on crops kept out of training entirely.')


# --- 5. export for the backend ---------------------------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)
np.savez(os.path.join(OUTPUT_DIR, 'ood_stats.npz'),
         centroids=centroids,
         centroid_classes=np.array(centroid_classes),
         threshold=np.float32(OOD_THRESHOLD),
         embed_layer=np.array('dense_hidden'),
         target_known_accept=np.float32(TARGET_KNOWN_ACCEPT))
print(f'\nSaved {OUTPUT_DIR}/ood_stats.npz  ->  copy to backend/Final_Model/')

## 16. Save Model Weights, Class Names & Open-Set Stats

In [ ]:
OUTPUT_DIR = '/content/model_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(PHASE2_WEIGHTS):
    shutil.copy(PHASE2_WEIGHTS, os.path.join(OUTPUT_DIR, 'best_model_phase2.weights.h5'))
    print('Copied best_model_phase2.weights.h5 from Drive to model_output/')
elif os.path.exists(PHASE1_WEIGHTS):
    print('WARNING: Only Phase 1 weights found. Phase 2 did not complete.')

with open(os.path.join(OUTPUT_DIR, 'class_names.json'), 'w') as f:
    json.dump(classes, f, indent=2)
print('Class names saved to model_output/class_names.json')

# The open-set centroids from section 15c. The backend needs this file to reject
# untrained crops; without it only the softmax heuristic applies.
if not os.path.exists(os.path.join(OUTPUT_DIR, 'ood_stats.npz')):
    print('WARNING: ood_stats.npz missing - run section 15c before saving, or the '
          'backend cannot reject untrained crops.')

# Keep a copy on Drive so a disconnect does not lose the finished artefacts.
DRIVE_OUT = os.path.join(DRIVE_DIR, 'model_output')
os.makedirs(DRIVE_OUT, exist_ok=True)

print('\nCopy ALL THREE to backend/Final_Model/ - they are one matched set:')
for f in ('best_model_phase2.weights.h5', 'class_names.json', 'ood_stats.npz'):
    p = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(p):
        shutil.copy(p, os.path.join(DRIVE_OUT, f))
        print(f'  {f:36s} {os.path.getsize(p) / 1024 / 1024:.2f} MB   (also on Drive)')
    else:
        print(f'  {f:36s} MISSING')

## 17. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT_DIR = '/content/model_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('Confusion matrix saved')

## 18. Training History Curves

In [ ]:
OUTPUT_DIR = '/content/model_output'
import matplotlib.pyplot as plt

if history1 is None and history2 is None:
    raise SystemExit('Nothing to plot: both phases were skipped (weights loaded '
                     'from a previous run). Delete the Drive checkpoints to retrain.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if history1 is not None:
    all_acc = history1.history['accuracy'] + history2.history['accuracy']
    all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
    all_loss = history1.history['loss'] + history2.history['loss']
    all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
    phase1_end = len(history1.history['accuracy'])
else:
    all_acc = history2.history['accuracy']
    all_val_acc = history2.history['val_accuracy']
    all_loss = history2.history['loss']
    all_val_loss = history2.history['val_loss']
    phase1_end = 0

axes[0].plot(all_acc, label='Train Accuracy')
axes[0].plot(all_val_acc, label='Val Accuracy')
if phase1_end > 0:
    axes[0].axvline(x=phase1_end, color='r', linestyle='--', alpha=0.5, label='Phase 2 start')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(all_loss, label='Train Loss')
axes[1].plot(all_val_loss, label='Val Loss')
if phase1_end > 0:
    axes[1].axvline(x=phase1_end, color='r', linestyle='--', alpha=0.5, label='Phase 2 start')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150)
plt.show()
print('Training history saved')

## 19. Download Output Files

In [ ]:
from google.colab import files

OUTPUT_DIR = '/content/model_output'

print('Output files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f:40s} {size / 1024 / 1024:.2f} MB')

print()
for f in ('best_model_phase2.weights.h5', 'class_names.json', 'ood_stats.npz'):
    p = os.path.join(OUTPUT_DIR, f)
    if os.path.exists(p):
        print(f'Downloading {f}...')
        files.download(p)
print('Done! Put all three in backend/Final_Model/.')